# AI Gateway Priority Routing — Test Suite

Tests for the JWT Citadel 3-priority PTU routing policy.

## Test Contracts (deployed via Bicep)

| Team | Priority | Model | TPM | PTU TPM | Monthly Quota |
|------|----------|-------|-----|---------|---------------|
| Alpha | P1 (production) | gpt-4.1-mini | 1,000 | 100 | 5,000 |
| Beta | P2 (standard) | gpt-4.1-mini | 400 | 200 | 3,000 |
| Gamma | P3 (economy) | gpt-4.1-mini | 300 | 0 | 1,000 |

## Prerequisites

1. Deploy with `azd up` (creates APIM, Entra apps, blob storage)
2. Run `add_app_reg_secrets.sh` to create client secrets
3. Fill in the config cell below with `azd env` values

In [ ]:
import requests
import json
import time
import msal
from dataclasses import dataclass, field
from typing import Optional
from IPython.display import display, HTML, Markdown
import subprocess

## Configuration

Load from `azd env` or fill manually.

In [ ]:
def get_azd_env():
    """Read all azd env values into a dict."""
    try:
        result = subprocess.run(['azd', 'env', 'get-values'], capture_output=True, text=True, check=True)
        env = {}
        for line in result.stdout.strip().split('\n'):
            if '=' in line:
                key, val = line.split('=', 1)
                env[key.strip()] = val.strip().strip('"')
        return env
    except Exception as e:
        print(f"Could not read azd env: {e}")
        return {}

azd = get_azd_env()

# Gateway config
GATEWAY_URL = azd.get('APIM_GATEWAY_URL', 'https://YOUR-APIM.azure-api.net')
API_URL = azd.get('APIM_API_URL', 'https://YOUR-APIM.azure-api.net/inference/openai')
TENANT_ID = azd.get('TENANT_ID', 'YOUR-TENANT-ID')
AUDIENCE = 'https://cognitiveservices.azure.com'

# Team credentials (from azd env after add_app_reg_secrets.sh)
TEAMS = {
    'alpha': {
        'client_id': azd.get('TEAM_ALPHA_APP_ID', ''),
        'client_secret': azd.get('TEAM_ALPHA_SECRET', ''),
        'priority': 1,
        'tpm': 1000,
        'ptu_tpm': 100,
        'monthly_quota': 7000,
    },
    'beta': {
        'client_id': azd.get('TEAM_BETA_APP_ID', ''),
        'client_secret': azd.get('TEAM_BETA_SECRET', ''),
        'priority': 2,
        'tpm': 400,
        'ptu_tpm': 200,
        'monthly_quota': 3000,
    },
    'gamma': {
        'client_id': azd.get('TEAM_GAMMA_APP_ID', ''),
        'client_secret': azd.get('TEAM_GAMMA_SECRET', ''),
        'priority': 3,
        'tpm': 300,
        'ptu_tpm': 0,
        'monthly_quota': 1000,
    },
}

MODEL = 'gpt-4.1-mini'
API_VERSION = '2025-03-01-preview'

print(f"Gateway: {GATEWAY_URL}")
print(f"API URL: {API_URL}")
print(f"Tenant:  {TENANT_ID}")
for name, t in TEAMS.items():
    print(f"  {name}: P{t['priority']} client_id={t['client_id'][:8]}... tpm={t['tpm']} ptu={t['ptu_tpm']} monthly={t['monthly_quota']}")

## Helper Functions

In [ ]:
_tokens = {}

def get_token(team_name: str) -> str:
    """Get a bearer token for a team using MSAL client credentials."""
    if team_name in _tokens:
        return _tokens[team_name]
    
    team = TEAMS[team_name]
    app = msal.ConfidentialClientApplication(
        client_id=team['client_id'],
        client_credential=team['client_secret'],
        authority=f'https://login.microsoftonline.com/{TENANT_ID}',
    )
    result = app.acquire_token_for_client(scopes=[f'{AUDIENCE}/.default'])
    if 'access_token' not in result:
        raise Exception(f"Token acquisition failed for {team_name}: {result.get('error_description', result)}")
    _tokens[team_name] = result['access_token']
    return _tokens[team_name]


@dataclass
class GatewayResponse:
    status: int
    team: str
    route_target: str = ''
    backend_type: str = ''
    caller_name: str = ''
    priority: str = ''
    tpm_limit: int = 0
    tpm_remaining: int = 0
    quota_limit: int = 0
    quota_remaining: int = 0
    ptu_utilization: str = ''
    ptu_consumed: int = 0
    ptu_limit: int = 0
    matched_identity: str = ''
    estimated_tokens: int = 0
    actual_tokens: int = 0
    is_streaming: bool = False
    body: dict = field(default_factory=dict)
    error: str = ''


def send_request(team_name: str, model: str = MODEL, prompt: str = 'Say hello in one word.',
                 max_tokens: int = 10) -> GatewayResponse:
    """Send a chat completion request through the gateway."""
    token = get_token(team_name)
    url = f"{API_URL}/deployments/{model}/chat/completions?api-version={API_VERSION}"
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }
    body = {
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
    }
    
    r = requests.post(url, headers=headers, json=body, timeout=30)
    h = r.headers
    
    def safe_int(val, default=0):
        try: return int(val)
        except: return default
    
    resp = GatewayResponse(
        status=r.status_code,
        team=team_name,
        route_target=h.get('x-route-target', ''),
        backend_type=h.get('x-backend-type', ''),
        caller_name=h.get('x-caller-name', ''),
        priority=h.get('x-caller-priority', ''),
        tpm_limit=safe_int(h.get('x-ratelimit-limit-tokens', 0)),
        tpm_remaining=safe_int(h.get('x-ratelimit-remaining-tokens', 0)),
        quota_limit=safe_int(h.get('x-quota-limit-tokens', 0)),
        quota_remaining=safe_int(h.get('x-quota-remaining-tokens', 0)),
        ptu_utilization=h.get('x-ptu-utilization', ''),
        ptu_consumed=safe_int(h.get('x-ptu-consumed', 0)),
        ptu_limit=safe_int(h.get('x-ptu-limit', 0)),
        matched_identity=h.get('x-caller-identity', ''),
        estimated_tokens=safe_int(h.get('x-estimated-tokens', 0)),
        actual_tokens=safe_int(h.get('x-actual-tokens', 0)),
        is_streaming=h.get('x-is-streaming', 'false') == 'true',
    )
    
    try:
        resp.body = r.json()
    except:
        resp.body = {'raw': r.text[:200]}
    
    if r.status_code >= 400:
        err = resp.body.get('error', '')
        msg = resp.body.get('message', '')
        if isinstance(err, dict):
            resp.error = err.get('message', str(err))
        elif msg:
            resp.error = f"{err}: {msg}" if err else msg
        else:
            resp.error = str(err) if err else r.text[:200]
    
    return resp


def send_streaming_request(team_name: str, model: str = MODEL,
                           prompt: str = 'Say hello in one word.',
                           max_tokens: int = 10) -> tuple[GatewayResponse, str]:
    """Send a streaming chat completion request. Returns (GatewayResponse, full_text)."""
    token = get_token(team_name)
    url = f"{API_URL}/deployments/{model}/chat/completions?api-version={API_VERSION}"
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }
    body = {
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'stream': True,
    }
    
    r = requests.post(url, headers=headers, json=body, timeout=30, stream=True)
    h = r.headers
    
    def safe_int(val, default=0):
        try: return int(val)
        except: return default
    
    # Collect streamed content
    full_text = ''
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue
        data = line[6:]
        if data == '[DONE]':
            break
        try:
            chunk = json.loads(data)
            delta = chunk.get('choices', [{}])[0].get('delta', {})
            content = delta.get('content', '')
            full_text += content
        except json.JSONDecodeError:
            pass
    
    resp = GatewayResponse(
        status=r.status_code,
        team=team_name,
        route_target=h.get('x-route-target', ''),
        backend_type=h.get('x-backend-type', ''),
        caller_name=h.get('x-caller-name', ''),
        priority=h.get('x-caller-priority', ''),
        tpm_limit=safe_int(h.get('x-ratelimit-limit-tokens', 0)),
        tpm_remaining=safe_int(h.get('x-ratelimit-remaining-tokens', 0)),
        quota_limit=safe_int(h.get('x-quota-limit-tokens', 0)),
        quota_remaining=safe_int(h.get('x-quota-remaining-tokens', 0)),
        ptu_utilization=h.get('x-ptu-utilization', ''),
        ptu_consumed=safe_int(h.get('x-ptu-consumed', 0)),
        ptu_limit=safe_int(h.get('x-ptu-limit', 0)),
        matched_identity=h.get('x-caller-identity', ''),
        estimated_tokens=safe_int(h.get('x-estimated-tokens', 0)),
        actual_tokens=safe_int(h.get('x-actual-tokens', 0)),
        is_streaming=h.get('x-is-streaming', 'false') == 'true',
        body={'streamed_text': full_text},
    )
    
    return resp, full_text


def print_response(r: GatewayResponse):
    icon = '✅' if r.status == 200 else '⚠️' if r.status == 429 else '🚫' if r.status == 403 else '❌'
    print(f"{icon} [{r.team}] {r.status} | route={r.route_target} backend={r.backend_type} pri={r.priority} | "
          f"tpm: {r.tpm_remaining}/{r.tpm_limit} | quota: {r.quota_remaining}/{r.quota_limit} | "
          f"ptu: {r.ptu_consumed}/{r.ptu_limit} util={r.ptu_utilization} | "
          f"est={r.estimated_tokens} actual={r.actual_tokens}")
    if r.error:
        print(f"   ↳ {r.error}")


def send_n(team: str, n: int, delay: float = 0.5, **kwargs) -> list[GatewayResponse]:
    """Send N requests for a team, printing each result."""
    results = []
    for i in range(n):
        r = send_request(team, **kwargs)
        print_response(r)
        results.append(r)
        if i < n - 1:
            time.sleep(delay)
    return results

---
## Test 1: Config Viewer

Verify the `/gateway/config` endpoint renders the contract configuration.

In [ ]:
config_url = f"{GATEWAY_URL}/gateway/config"
r = requests.get(config_url)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
if r.status_code == 200:
    display(HTML(r.text))
else:
    print(r.text[:500])

In [ ]:
config_url = f"{GATEWAY_URL}/gateway/config/refresh"
r = requests.post(config_url)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
if r.status_code == 200:
    display(HTML(r.text))
else:
    print(r.text[:500])

---
## Test 2: Identity & Contract Resolution

Each team should be recognized and get correct contract fields in response headers.

In [ ]:
print("=== Identity Resolution ===")
for team in [ 'alpha', 'beta','gamma']:
    r = send_request(team)
    print_response(r)
    expected = TEAMS[team]
    assert r.status == 200, f"{team}: expected 200, got {r.status}: {r.error}"
    assert r.tpm_limit == expected['tpm'], f"{team}: tpm_limit {r.tpm_limit} != {expected['tpm']}"
    assert r.quota_limit == expected['monthly_quota'], f"{team}: quota_limit {r.quota_limit} != {expected['monthly_quota']}"
    assert r.ptu_limit == expected['ptu_tpm'], f"{team}: ptu_limit {r.ptu_limit} != {expected['ptu_tpm']}"
    print(f"   ✓ Contract verified: name={r.caller_name} identity={r.matched_identity}\n")

print("All identity tests passed! ✅")

---
## Test 3: Priority Routing

Three routing tiers:
- **P1 (Alpha)**: priority=production, route=pool (PTU first) or payg-overflow (PTU soft limit reached)
- **P2 (Beta)**: priority=standard, route=pool when PTU idle, payg-overflow when busy
- **P3 (Gamma)**: priority=economy, route=payg (always PAYG-only pool)

In [ ]:
print("=== Priority Routing & Token Counters ===\n")

# Expected routes:
#   P1→pool (PTU first) or payg-overflow (PTU soft limit reached)
#   P2→pool or payg-overflow (depends on utilization + PTU quota)
#   P3→payg (always PAYG-only)
expected_routes = {1: ('pool', 'payg-overflow'), 2: ('pool', 'payg-overflow'), 3: 'payg'}

for team_name, label in [('alpha', 'P1/production'), ('beta', 'P2/standard'), ('gamma', 'P3/economy')]:
    expected = TEAMS[team_name]
    exp_priority = {1: 'production', 2: 'standard', 3: 'economy'}[expected['priority']]
    print(f"--- {team_name.upper()} ({label}) ---")

    results = []
    for i in range(3):
        r = send_request(team_name)
        print_response(r)
        results.append(r)
        assert r.status == 200, f"{team_name} req {i+1}: expected 200, got {r.status}: {r.error}"
        exp_route = expected_routes[expected['priority']]
        if isinstance(exp_route, tuple):
            assert r.route_target in exp_route, f"{team_name}: expected route in {exp_route}, got {r.route_target}"
        else:
            assert r.route_target == exp_route, f"{team_name}: expected route={exp_route}, got {r.route_target}"
        assert r.priority == exp_priority, f"{team_name}: expected priority={exp_priority}, got {r.priority}"
        assert r.tpm_limit == expected['tpm'], f"{team_name}: tpm_limit {r.tpm_limit} != {expected['tpm']}"
        assert r.quota_limit == expected['monthly_quota'], f"{team_name}: quota_limit {r.quota_limit} != {expected['monthly_quota']}"
        assert r.ptu_limit == expected['ptu_tpm'], f"{team_name}: ptu_limit {r.ptu_limit} != {expected['ptu_tpm']}"
        time.sleep(0.3)

    # Verify overall counter movement (first vs last)
    first, last = results[0], results[-1]
    assert last.tpm_remaining < first.tpm_remaining, \
        f"{team_name}: tpm_remaining should decrease over 3 requests ({first.tpm_remaining} → {last.tpm_remaining})"
    assert last.quota_remaining < first.quota_remaining, \
        f"{team_name}: quota_remaining should decrease over 3 requests ({first.quota_remaining} → {last.quota_remaining})"

    print(f"   ✓ {team_name}: route={results[0].route_target} pri={exp_priority} | tpm: {first.tpm_remaining}→{last.tpm_remaining} "
          f"quota: {first.quota_remaining}→{last.quota_remaining}\n")

print("All routing & counter tests passed! ✅")

---
## Test 4: Model Access Control

Teams should only access models in their contract. A request for an unlisted model should get 403.

In [ ]:
print("=== Model Access Control ===")

# Alpha: gpt-4.1-mini allowed
r = send_request('alpha', model='gpt-4.1-mini')
print_response(r)
assert r.status == 200, f"Alpha/gpt-4.1-mini should be 200, got {r.status}"
print("   ✓ Alpha → gpt-4.1-mini allowed\n")

# Alpha: gpt-4o should be denied (not in test contract)
r = send_request('alpha', model='gpt-4o')
print_response(r)
assert r.status == 403, f"Alpha/gpt-4o should be 403, got {r.status}"
assert 'model_not_authorized' in str(r.body), f"Expected model_not_authorized error"
print("   ✓ Alpha → gpt-4o denied\n")

print("Model access tests passed! ✅")

---
## Test 5: Unknown Caller Rejection

A token from an unregistered app should get 401.

In [ ]:
print("=== Unknown Caller ===")

url = f"{API_URL}/deployments/{MODEL}/chat/completions?api-version={API_VERSION}"
r = requests.post(url, headers={'Authorization': 'Bearer invalid-token', 'Content-Type': 'application/json'},
                  json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=10)
print(f"Status: {r.status_code}")
assert r.status_code == 401, f"Invalid token should get 401, got {r.status_code}"
print("   ✓ Invalid token → 401\n")

print("Unknown caller tests passed! ✅")

---
## Test 6: Per-Model TPM Rate Limiting

Gamma has only 300 TPM for gpt-4o. Rapid requests should trigger 429.

In [ ]:
print("=== Per-Model TPM Rate Limit (Gamma: 300 TPM) ===")
print("Sending rapid requests to exhaust TPM...\n")

big_prompt = 'Write a detailed essay about the history of computing. ' * 5

results = send_n('gamma', 15, delay=0.2, prompt=big_prompt)

success = sum(1 for r in results if r.status == 200)
rate_limited = sum(1 for r in results if r.status == 429)

print(f"\nResults: {success} success, {rate_limited} rate-limited")
assert rate_limited > 0, "Expected at least one 429 with 300 TPM limit"
print("\nTPM rate limiting works! ✅")

---
## Test 7: Monthly Quota Exhaustion

Gamma has only 1,000 monthly quota. Keep sending until exhausted.

In [ ]:
print("=== Monthly Quota Exhaustion (Gamma: 1000 tokens) ===")
print("Sending requests until quota is exhausted...\n")

quota_hit = False
for i in range(50):
    r = send_request('gamma', prompt='Tell me a long story about dragons and knights. ' * 3)
    print_response(r)
    # llm-token-limit returns 403 (not 429) when monthly quota is exceeded
    if r.status == 403 and 'quota' in r.error.lower():
        print(f"\n   ✓ Monthly quota exhausted after {i+1} requests (403 — quota exceeded)")
        quota_hit = True
        break
    if r.status == 429 and r.quota_remaining <= 0:
        print(f"\n   ✓ Monthly quota exhausted after {i+1} requests (429 — rate limited)")
        quota_hit = True
        break
    time.sleep(1)  # wait for TPM window to reset between batches

if quota_hit:
    print("Monthly quota limiting works! ✅")
else:
    print("⚠️ Quota not exhausted in 50 requests — may need more/larger prompts")

---
## Test 8: PTU → PAYG → PTU Cycle (Extended Run)

Alpha (P1) has 100 PTU TPM soft limit. This test runs for **~90 seconds** to observe the full lifecycle:

1. **PTU phase**: Requests start on PTU while `x-ptu-consumed` < `x-ptu-limit` (100)
2. **PAYG overflow phase**: Once PTU soft limit is crossed, routing switches to PAYG-only pool
3. **PTU recovery phase**: After the 60s TPM window resets, PTU consumed drops and requests return to PTU

Headers tracked per request:
- **`x-backend-type`**: `ptu` or `payg` — which deployment served the request
- **`x-ptu-consumed`** / **`x-ptu-limit`**: running PTU token counter vs soft cap

In [ ]:
print("=== PTU → PAYG → PTU Cycle (Alpha: 100 PTU TPM, ~90s run) ===")
print("Sending requests for ~90 seconds to observe full PTU lifecycle...\n")
# Show Alpha's contract limits upfront
alpha = TEAMS['alpha']
print(f"Alpha contract limits:")
print(f"  TPM (per-model):  {alpha['tpm']}")
print(f"  PTU TPM (soft):   {alpha['ptu_tpm']}")
print(f"  Monthly quota:    {alpha['monthly_quota']}")
print()

DURATION_SECONDS = 90
DELAY = 1.0  # 1s between requests — enough to see window effects

results = []
start_time = time.time()

while time.time() - start_time < DURATION_SECONDS:
    elapsed = time.time() - start_time
    r = send_request('alpha', prompt='Explain quantum computing in detail. ' * 3)
    r._elapsed = elapsed  # stash timing for the summary
    results.append(r)
    icon = '🔵' if r.backend_type == 'ptu' else '🟡' if r.backend_type == 'payg' else '⚪'
    print(f"  {icon} t={elapsed:5.1f}s  backend={r.backend_type:4s}  ptu={r.ptu_consumed:4d}/{r.ptu_limit}  "
          f"tpm={r.tpm_remaining:4d}/{r.tpm_limit}  quota={r.quota_remaining}/{r.quota_limit}  "
          f"est={r.estimated_tokens} actual={r.actual_tokens}  status={r.status}")
    if r.status != 200:
        print(f"       ↳ {r.error}")
    time.sleep(DELAY)

actual_duration = time.time() - start_time

# ── Summary ──
success = sum(1 for r in results if r.status == 200)
failed = sum(1 for r in results if r.status != 200)
ptu_count = sum(1 for r in results if r.backend_type == 'ptu')
payg_count = sum(1 for r in results if r.backend_type == 'payg')
peak_consumed = max((r.ptu_consumed for r in results), default=0)
ptu_limit = results[0].ptu_limit if results else 0
total_actual = sum(r.actual_tokens for r in results if r.status == 200)
total_estimated = sum(r.estimated_tokens for r in results if r.status == 200)

print(f"\n{'='*80}")
print(f"Duration: {actual_duration:.0f}s | Requests: {len(results)} ({success} ok, {failed} failed)")
print(f"Backend breakdown: {ptu_count} PTU 🔵, {payg_count} PAYG 🟡")
print(f"PTU peak consumed: {peak_consumed} / {ptu_limit} limit")
print(f"Total tokens — estimated: {total_estimated}, actual: {total_actual}")
print(f"Quota remaining: {results[-1].quota_remaining} / {results[-1].quota_limit} monthly")
print(f"TPM remaining:   {results[-1].tpm_remaining} / {results[-1].tpm_limit}")
print(f"{'='*80}")

# ── Timeline ──
print(f"\nTimeline (🔵=PTU  🟡=PAYG  🔴=error):")
for r in results:
    if r.status != 200:
        icon = '🔴'
    elif r.backend_type == 'ptu':
        icon = '🔵'
    else:
        icon = '🟡'
    bar_pct = min(r.ptu_consumed / r.ptu_limit * 100, 100) if r.ptu_limit > 0 else 0
    bar = '█' * int(bar_pct / 5) + '░' * (20 - int(bar_pct / 5))
    over = " ← OVER" if r.ptu_consumed > r.ptu_limit and r.ptu_limit > 0 else ""
    print(f"  {icon} t={r._elapsed:5.1f}s  {r.backend_type:4s}  [{bar}] {r.ptu_consumed:4d}/{r.ptu_limit}{over}")

# ── Phase detection ──
phases = []
current_phase = results[0].backend_type if results else None
phase_start = 0
for i, r in enumerate(results):
    bt = r.backend_type if r.status == 200 else 'error'
    if bt != current_phase:
        phases.append((current_phase, phase_start, i - 1))
        current_phase = bt
        phase_start = i
phases.append((current_phase, phase_start, len(results) - 1))

print(f"\nPhases detected:")
for phase_type, start_idx, end_idx in phases:
    t_start = results[start_idx]._elapsed
    t_end = results[end_idx]._elapsed
    count = end_idx - start_idx + 1
    label = {'ptu': '🔵 PTU', 'payg': '🟡 PAYG', 'error': '🔴 Error'}.get(phase_type, phase_type)
    print(f"  {label}: requests #{start_idx+1}-{end_idx+1} (t={t_start:.0f}s–{t_end:.0f}s, {count} requests)")

phase_types = [p[0] for p in phases]

# Check for the full cycle
if phase_types == ['ptu', 'payg', 'ptu'] or (len(phase_types) >= 3 and 'ptu' in phase_types and 'payg' in phase_types):
    saw_return = any(p[0] == 'ptu' for p in phases[2:]) if len(phases) > 2 else False
    if saw_return or phase_types[-1] == 'ptu':
        print(f"\n   ✓ Full PTU → PAYG → PTU cycle observed!")
        print("PTU recovery after window reset verified! ✅")
    else:
        print(f"\n   ✓ PTU → PAYG overflow observed (no recovery yet — may need longer run)")
        print("PTU overflow verified! ✅")
elif 'payg' in phase_types and 'ptu' in phase_types:
    print(f"\n   ✓ Both PTU and PAYG backends used")
    print("Backend switching observed! ✅")
elif ptu_count == len(results):
    print(f"\n   ℹ️ All requests stayed on PTU — real PTU capacity not exhausted")
    print("   The soft PTU limit is informational; circuit breaker only trips on real 429")
    print("PTU tracking verified ✅")
else:
    print(f"\nBackend distribution noted ✅")

---
## Test 9: P2 Routing Under PTU Load

P2 (standard) gets PTU only when **global utilization < 50%** AND team has PTU quota.
When P1 floods PTU, the pre-debit cache should push utilization above the threshold,
causing P2 to shift from `pool` → `payg-overflow`.

In [ ]:
print("=== P2 Routing Under PTU Load ===")
print("Step 1: Wait 65s for PTU cache to expire (60s TTL)...\n")
time.sleep(65)

print("Step 2: Baseline — P2 should route to pool at low utilization\n")
r = send_request('beta')
print_response(r)
baseline_route = r.route_target
baseline_util = r.ptu_utilization
print(f"   Baseline: route={baseline_route}, util={baseline_util}")

print(f"\nStep 3: P1 (Alpha) flooding PTU to raise utilization via pre-debit...\n")
flood_results = send_n('alpha', 10, delay=0.3, prompt='Explain everything about machine learning. ' * 5)

print(f"\nStep 4: Check P2 routing after P1 flood...\n")
time.sleep(1)

p2_results = []
for i in range(5):
    r = send_request('beta')
    print_response(r)
    p2_results.append(r)
    time.sleep(0.5)

pool_routes = sum(1 for r in p2_results if r.route_target == 'pool')
overflow_routes = sum(1 for r in p2_results if r.route_target == 'payg-overflow')

print(f"\nBaseline: route={baseline_route} util={baseline_util}")
print(f"After flood: {pool_routes} pool, {overflow_routes} payg-overflow")
print(f"Final utilization: {p2_results[-1].ptu_utilization}")

if baseline_route == 'pool' and overflow_routes > 0:
    print("\n   ✓ P2 shifted from pool → payg-overflow as utilization rose above 50%")
    print("P2 dynamic routing works! ✅")
elif overflow_routes == len(p2_results):
    print("\n   P2 went straight to payg-overflow — utilization was already high")
    print("P2 PAYG routing works! ✅")
else:
    print(f"\n   P2 stayed on pool — utilization ({p2_results[-1].ptu_utilization}) may not have crossed 50%")
    print("   Consider increasing flood volume or lowering threshold")

---
## Test 10: Cross-Team Isolation

Rate-limiting one team should NOT affect another team's quota.

In [ ]:
print("=== Cross-Team Isolation ===")
print("Step 1: Exhaust Gamma's TPM...\n")

big_prompt = 'Explain everything. ' * 10
gamma_results = send_n('gamma', 10, delay=0.1, prompt=big_prompt)
gamma_429 = sum(1 for r in gamma_results if r.status == 429)
print(f"\nGamma: {gamma_429} rate-limited\n")

print("Step 2: Alpha should still work fine...\n")
r = send_request('alpha')
print_response(r)
assert r.status == 200, f"Alpha should not be affected by Gamma's rate limit, got {r.status}"
print("\n   ✓ Alpha unaffected by Gamma's rate limiting")

print("\nStep 3: Beta should still work fine...\n")
r = send_request('beta')
print_response(r)
assert r.status == 200, f"Beta should not be affected by Gamma's rate limit, got {r.status}"
print("\n   ✓ Beta unaffected by Gamma's rate limiting")

print("\nCross-team isolation verified! ✅")

---
## Test 11: Streaming Support

Validate that streaming requests work through the gateway:
- Response headers should include `x-is-streaming: true`
- SSE chunks should arrive and contain content
- PTU pre-debit estimate should stand (no reconciliation for streaming)

In [ ]:
print("=== Streaming Support ===\n")

# Test 1: Streaming request returns content
print("Step 1: Send streaming request (Alpha)\n")
r_stream, text = send_streaming_request('alpha', prompt='Count from 1 to 5.', max_tokens=50)
print(f"   Status: {r_stream.status}")
print(f"   Route: {r_stream.route_target}")
print(f"   Backend: {r_stream.backend_type}")
print(f"   Is-streaming: {r_stream.is_streaming}")
print(f"   Estimated tokens: {r_stream.estimated_tokens}")
print(f"   Actual tokens: {r_stream.actual_tokens}")
print(f"   Streamed text: '{text[:100]}...'")
assert r_stream.status == 200, f"Streaming request failed: {r_stream.status}"
assert r_stream.is_streaming == True, f"Expected x-is-streaming=true, got {r_stream.is_streaming}"
assert len(text) > 0, "No content received from streaming response"
print("   ✓ Streaming request returned content")

# For streaming, actual_tokens should be 0 (can't parse SSE body in outbound)
# The estimate stands as the PTU consumption value
print(f"\n   ℹ️ actual_tokens={r_stream.actual_tokens} (expected 0 for streaming — estimate stands)")

# Test 2: Non-streaming for comparison
print("\nStep 2: Compare with non-streaming request\n")
r_normal = send_request('alpha', prompt='Count from 1 to 5.', max_tokens=50)
print(f"   Status: {r_normal.status}")
print(f"   Is-streaming: {r_normal.is_streaming}")
print(f"   Estimated tokens: {r_normal.estimated_tokens}")
print(f"   Actual tokens: {r_normal.actual_tokens}")
assert r_normal.is_streaming == False, f"Non-streaming should have x-is-streaming=false"
if r_normal.actual_tokens > 0:
    print(f"   ✓ Non-streaming has actual tokens for reconciliation")
    delta = r_normal.actual_tokens - r_normal.estimated_tokens
    print(f"   Estimation delta: {delta:+d} tokens ({r_normal.estimated_tokens} est vs {r_normal.actual_tokens} actual)")

print("\nStreaming tests passed! ✅")

---
## Test 12: Token Estimation Accuracy

Measure how well the inbound token estimate correlates with actual `usage.total_tokens`.
The policy estimates: `prompt_chars/4 + max_tokens/2`.

Sends requests with varying prompt sizes and max_tokens to profile the estimation error.
This data helps calibrate the heuristic.

In [ ]:
print("=== Token Estimation Accuracy ===\n")

# Varying prompt sizes and max_tokens to profile estimation
test_cases = [
    ('tiny',   'Hi.',                                          10),
    ('short',  'Say hello.',                                   10),
    ('medium', 'Explain quantum computing in one paragraph.',  50),
    ('long',   'Write a detailed essay about AI. ' * 10,       100),
    ('xl',     'Analyze the impact of technology. ' * 50,      10),
    ('high-mt','Tell me a joke.',                              200),
]

print(f"{'Case':<10} {'Prompt chars':>12} {'Max tokens':>10} {'Estimated':>10} {'Actual':>8} {'Delta':>8} {'Error%':>8}")
print("-" * 68)

deltas = []
for label, prompt, mt in test_cases:
    r = send_request('alpha', prompt=prompt, max_tokens=mt)
    if r.status != 200:
        print(f"{label:<10} FAILED ({r.status})")
        continue
    est = r.estimated_tokens
    act = r.actual_tokens
    delta = act - est
    err_pct = (delta / act * 100) if act > 0 else 0
    deltas.append((label, len(prompt), mt, est, act, delta, err_pct))
    print(f"{label:<10} {len(prompt):>12} {mt:>10} {est:>10} {act:>8} {delta:>+8} {err_pct:>+7.1f}%")
    time.sleep(0.5)

if deltas:
    avg_err = sum(abs(d[5]) for d in deltas) / len(deltas)
    avg_pct = sum(abs(d[6]) for d in deltas) / len(deltas)
    over_est = sum(1 for d in deltas if d[5] < 0)
    under_est = sum(1 for d in deltas if d[5] > 0)
    print(f"\n{'='*68}")
    print(f"Mean absolute error: {avg_err:.0f} tokens ({avg_pct:.1f}%)")
    print(f"Over-estimated: {over_est}/{len(deltas)} | Under-estimated: {under_est}/{len(deltas)}")
    print(f"{'='*68}")
    print(f"\nℹ️ Over-estimation is conservative (sends fewer requests to PTU than needed)")
    print(f"   Under-estimation is risky (allows more PTU usage than soft limit)")

print("\nEstimation profiling complete ✅")

---
## Test Summary

In [ ]:
summary = """
| # | Test | What it verifies |
|---|------|------------------|
| 1 | Config Viewer | `/gateway/config` renders HTML with contract details |
| 2 | Identity Resolution | Each team gets correct contract (tpm, quota, ptu limits) |
| 3 | Priority Routing | P1→pool, P2→pool/overflow, P3→payg, counters decrease |
| 4 | Model Access Control | Unlisted models get 403 |
| 5 | Unknown Caller | Invalid/unregistered tokens get 401 |
| 6 | Per-Model TPM | Rapid requests hit 429 at TPM limit |
| 7 | Monthly Quota | Cumulative tokens exhaust monthly cap |
| 8 | PTU → PAYG Cycle | PTU consumed → pre-debit routes to PAYG → cache resets → back to PTU |
| 9 | P2 Under Load | P1 flood raises utilization → P2 shifts pool → payg-overflow |
| 10 | Cross-Team Isolation | One team's rate limit doesn't affect others |
| 11 | Streaming | `stream:true` works, `x-is-streaming` header, estimate stands |
| 12 | Estimation Accuracy | Profile estimated vs actual tokens across prompt sizes |
| 13 | Header Completeness | All custom gateway headers present on 200 and error responses |
"""
display(Markdown(summary))

---
## Test 13: Response Header Completeness

Verify that all expected custom gateway headers are present and non-empty on a successful 200 response.

In [ ]:
print("=== Response Header Completeness ===\n")

# Expected headers on a 200 response
EXPECTED_200_HEADERS = [
    'x-ratelimit-limit-tokens',
    'x-ratelimit-remaining-tokens',
    'x-quota-limit-tokens',
    'x-quota-remaining-tokens',
    'x-caller-name',
    'x-caller-identity',
    'x-caller-priority',
    'x-route-target',
    'x-backend-type',
    'x-ptu-utilization',
    'x-ptu-consumed',
    'x-ptu-limit',
    'x-actual-tokens',
    'x-estimated-tokens',
    'x-is-streaming',
    'x-tokens-consumed',
]

# Expected headers on a 401/403/on-error response
EXPECTED_ERROR_HEADERS = [
    'x-error-reason',
    'x-error-message',
    'x-error-section',
    'x-error-source',
]

# --- Test 200 headers ---
print("Step 1: Verify all headers present on 200 response (Alpha)\n")
token = get_token('alpha')
url = f"{API_URL}/deployments/{MODEL}/chat/completions?api-version={API_VERSION}"
r = requests.post(url, headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
                  json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=30)
assert r.status_code == 200, f"Expected 200, got {r.status_code}"

missing = []
empty = []
for h in EXPECTED_200_HEADERS:
    val = r.headers.get(h)
    if val is None:
        missing.append(h)
    elif val.strip() == '':
        empty.append(h)
    else:
        print(f"   ✓ {h}: {val}")

assert not missing, f"Missing headers: {missing}"
assert not empty, f"Empty headers: {empty}"
print(f"\n   All {len(EXPECTED_200_HEADERS)} headers present and non-empty ✅")

# --- Test error headers ---
print("\nStep 2: Verify error headers on 401 response (invalid token)\n")
r2 = requests.post(url, headers={'Authorization': 'Bearer invalid-token', 'Content-Type': 'application/json'},
                   json={'messages': [{'role': 'user', 'content': 'test'}], 'max_tokens': 5}, timeout=10)
assert r2.status_code == 401, f"Expected 401, got {r2.status_code}"

missing_err = []
for h in EXPECTED_ERROR_HEADERS:
    val = r2.headers.get(h)
    if val is None:
        missing_err.append(h)
    else:
        print(f"   ✓ {h}: {val}")

assert not missing_err, f"Missing error headers: {missing_err}"
print(f"\n   All {len(EXPECTED_ERROR_HEADERS)} error headers present ✅")

print("\nHeader completeness tests passed! ✅")